<a href="https://colab.research.google.com/github/Chandan0731/pyspark-movie-reviews/blob/main/imdb_pyspark_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 0. Unzip the uploaded Kaggle dataset
!unzip -q -o "*.zip"

# 1. Install PySpark
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, length

# 2. Initialize Spark & Load CSV
spark = SparkSession.builder.appName("IMDb_Movie_Reviews_Analysis").getOrCreate()

df = spark.read.csv(
    "IMDB Dataset.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

# 3. Explore the Data
print("--- Schema ---")
df.printSchema()

print("--- First 5 Rows ---")
df.show(5, truncate=60)

# 4. Clean the Data (remove null or empty reviews)
df_cleaned = df.filter(col("review").isNotNull() & (trim(col("review")) != ""))

# 5. Analyze the Data
print(f"Total Valid Reviews: {df_cleaned.count()}")
df_cleaned.groupBy("sentiment").count().show()

# 6. Transform the Data (add review length & filter > 500 chars)
df_transformed = df_cleaned.withColumn("review_length", length(col("review")))
df_long = df_transformed.filter(col("review_length") > 500)
print(f"Reviews > 500 characters: {df_long.count()}")
df_long.select("sentiment", "review_length", "review").show(5, truncate=50)

# 7. Save Cleaned & Transformed Results as CSV
df_transformed.coalesce(1).write.mode("overwrite").option("header", "true").csv("cleaned_imdb_reviews")
print("Saved results successfully!")

--- Schema ---
root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)

--- First 5 Rows ---
+------------------------------------------------------------+---------+
|                                                      review|sentiment|
+------------------------------------------------------------+---------+
|One of the other reviewers has mentioned that after watch...| positive|
|A wonderful little production. <br /><br />The filming te...| positive|
|I thought this was a wonderful way to spend time on a too...| positive|
|Basically there's a family where a little boy (Jake) thin...| negative|
|Petter Mattei's "Love in the Time of Money" is a visually...| positive|
+------------------------------------------------------------+---------+
only showing top 5 rows
Total Valid Reviews: 50000
+---------+-----+
|sentiment|count|
+---------+-----+
| positive|25000|
| negative|25000|
+---------+-----+

Reviews > 500 characters: 45017
+---------+-------------+-----

In [2]:
import glob, shutil

# Grab the single CSV file Spark generated inside cleaned_imdb_reviews/
part_csv = glob.glob("cleaned_imdb_reviews/*.csv")[0]
shutil.copy(part_csv, "cleaned_transformed_imdb_reviews.csv")

# Zip it so it fits under GitHub's 25 MB web upload limit
!zip -q cleaned_transformed_imdb_reviews.zip cleaned_transformed_imdb_reviews.csv
print("Ready to download: cleaned_transformed_imdb_reviews.zip")

Ready to download: cleaned_transformed_imdb_reviews.zip
